# Tarea 11 · Correlaciones de Pearson y Spearman

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/11_correlaciones.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Matías.

Comparamos la relación entre la antigüedad (`tenure`), el cargo mensual (`MonthlyCharges`) y el cargo acumulado (`TotalCharges`). Calculamos dos matrices que sirven de entrada para el heatmap del punto 18.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repo y ejecutá todas las celdas.
- **Colab:** ejecutá todas las celdas. El notebook lee los datos desde GitHub. El botón apunta a `main` y estará disponible cuando se integre la tarea. Para probarla antes, subí el archivo `.ipynb` a Colab.

## 1. Carga del dataset

Usamos `telco_clean.csv` cuando está disponible. Mientras tanto, leemos `telco_provisorio.csv` y mostramos la fuente utilizada. Para cada par de variables, Pandas usa las filas que tienen los dos valores.

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

variables = ["tenure", "MonthlyCharges", "TotalCharges"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 2. Correlación de Pearson

Pearson mide qué tan cerca está la relación entre dos variables de seguir una línea recta. El coeficiente va de −1 a +1. Un signo positivo indica que los valores altos de una variable tienden a aparecer junto con valores altos de la otra. Calculamos la matriz y contamos cuántas filas se usaron en cada comparación.

In [ ]:
pearson = datos.corr(method="pearson")
presentes = datos.notna().astype(int)
pares_validos = presentes.T @ presentes

print("Correlación de Pearson")
print(pearson.round(3).to_string())
print("\nFilas disponibles por par de variables")
print(pares_validos.to_string())

Con el CSV provisorio, la asociación lineal más alta es la de antigüedad y cargo acumulado (0,826). Los clientes con más meses de servicio tienden a haber acumulado más cargos. Los pares que incluyen `TotalCharges` usan 7.032 filas, porque esa columna tiene 11 faltantes en el CSV provisorio.

## 3. Correlación de Spearman

Spearman trabaja con los rangos de los valores. Mide si, al ordenar a los clientes por una variable, la otra tiende a subir o bajar en el mismo orden, incluso si la relación forma una curva. Usamos las mismas tres columnas para comparar las dos matrices.

In [ ]:
spearman = datos.corr(method="spearman")
print("Correlación de Spearman")
print(spearman.round(3).to_string())

## 4. Comparación e interpretación

- **Antigüedad y cargo acumulado.** Pearson da 0,826 y Spearman da 0,889. Los clientes con mayor antigüedad suelen ocupar posiciones más altas en el cargo acumulado. La correlación es más alta al comparar el orden que al ajustar la relación a una línea recta.
- **Cargo mensual y cargo acumulado.** Pearson da 0,651 y Spearman da 0,638. Los clientes con cargos mensuales más altos tienden a acumular más cargos, aunque el total también varía con los meses de servicio.
- **Antigüedad y cargo mensual.** Pearson da 0,248 y Spearman da 0,276. Es la asociación positiva más débil. Saber cuántos meses lleva un cliente da poca información sobre cuánto paga por mes.

Estas cifras corresponden al CSV provisorio. Revisarlas con las matrices actualizadas cuando se use `telco_clean.csv`. Las correlaciones describen asociaciones entre estas variables y, por sí solas, no establecen causalidad.